# Évaluation officielle multi-vs-multi — VeRi-776 (3 méthodes)

**Ré-identification multi-vs-multi de véhicules** sur le dataset **VeRi-776** :
pour chaque véhicule, un **ensemble de requête multi-caméras** est comparé à un
**ensemble de galerie multi-caméras**, chacun fusionné en **un embedding robuste**
puis comparé **identité vs identité** (vérification 1-to-1 + identification).

## Les 3 méthodes évaluées (ordre officiel de la mission)
| # | Méthode | Principe |
|---|---|---|
| 1 | **Averaging** (baseline) | Moyenne simple des embeddings L2 |
| 2 | **VCNet** | Fusion **pondérée par point de vue** (confiance de vue × complémentarité) |
| 3 | **EV-MoE / CAFNet** | **Mixture of Experts + Attention** : gating par vue → experts → self-attention |

## Plan du notebook
1. **Configuration** : GPU, dépendances, repo officiel **FastReID**, poids VeRi, dataset
2. **Protocole multi-vs-multi** : découpage requête/galerie par caméras
3. **Extraction des embeddings** FastReID SBS R50-ibn (2048-D) sur GPU
4. **Les 3 méthodes de fusion** (implémentation officielle)
5. **Entraînement d'EV-MoE** sur GPU (perte CE + triplet, gating par vue réelle)
6. **Évaluation officielle** : Rank-1/5/10, mAP, Acc 1:1, AUC, courbes CMC/ROC
7. **Export ZIP** : rapport JSON + TXT + figures + modèle entraîné

##  Prérequis (à faire avant de lancer)
- **Dataset VeRi-776** dans l'input Kaggle (dossier contenant `image_train/`,
  `image_test/`, `image_query/`, `name_test.txt`, `test_label.xml`, `train_label.xml`).
- **Accélérateur = GPU** (T4/P100) dans les paramètres du notebook.
- Internet activé (pour cloner FastReID et télécharger les poids officiels).

---
## Partie 0 — Configuration (GPU, dépendances, FastReID, données)

### 0.1 Dépendances + vérification du GPU

### 0.2 Récupération du repo officiel **FastReID** (JDAI-CV/fast-reid)

On clone le dépôt officiel (les `configs/VeRi/sbs_R50-ibn.yml` et le package
`fastreid` en font partie) puis on l'ajoute au `sys.path`. Aucune compilation
n'est nécessaire : FastReID est utilisé uniquement en **inférence**.

In [9]:
!pip install -q yacs termcolor tabulate portalocker iopath fvcore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 6.4 MB/s eta 0:00:00


In [10]:
import collections, collections.abc, sys, types
for _n in ('Mapping','MutableMapping','Sequence','Iterable','Callable',
           'Container','Hashable','Sized','Set','KeysView','ItemsView','ValuesView'):
    if not hasattr(collections, _n):
        setattr(collections, _n, getattr(collections.abc, _n))

if not hasattr(__import__('torch'), '_six'):
    _m = types.ModuleType('torch._six')
    _m.string_classes = str
    _m.container_abcs = collections.abc
    sys.modules['torch._six'] = _m
    __import__('torch')._six = _m

print('✔ Correctif compatibilité appliqué')

✔ Correctif compatibilité appliqué


In [11]:
# =============================================================================
# 0.2 Clone du repo officiel FastReID + import
# =============================================================================
import os
import subprocess
import sys

FASTREID_DIR = '/kaggle/working/FastReID'
if not os.path.isdir(os.path.join(FASTREID_DIR, 'fastreid')):
    print('Clonage du dépôt officiel FastReID (JDAI-CV/fast-reid)...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/JDAI-CV/fast-reid.git', FASTREID_DIR],
                   check=True)
else:
    print('FastReID déjà présent :', FASTREID_DIR)

sys.path.insert(0, FASTREID_DIR)

from fastreid.config import get_cfg            # noqa: E402
from fastreid.engine import DefaultPredictor   # noqa: E402
print('✔ Import FastReID OK')

FastReID déjà présent : /kaggle/working/FastReID
✔ Import FastReID OK


In [12]:
# =============================================================================
# 0.1 Installation des dépendances minimales + vérification du GPU
# =============================================================================
import subprocess
import sys


def pip_install(pkg):
    print('$ pip install', pkg)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)


for pkg in ['yacs', 'termcolor', 'tabulate', 'tensorboard', 'scipy']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        pip_install(pkg)
try:
    import faiss  # noqa: F401
except ImportError:
    pip_install('faiss-cpu')

import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch :', torch.__version__)
print('Device  :', DEVICE)
if DEVICE == 'cuda':
    print('GPU     :', torch.cuda.get_device_name(0), '| VRAM',
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'Go')
assert DEVICE == 'cuda', (
    'Active le GPU : Éditeur → Paramètres du notebook → Accélérateur = GPU (T4/P100)')

PyTorch : 2.10.0+cu128
Device  : cuda
GPU     : Tesla T4 | VRAM 15.6 Go


### 0.3 Poids officiels FastReID entraînés sur **VeRi-776**

Le model zoo officiel FastReID fournit `veri_sbs_R50-ibn.pth` (SBS R50-ibn
entraîné sur VeRi, Rank-1 97,0 %). Si un `.pth` est déjà présent dans l'input
Kaggle, il est utilisé en priorité ; sinon téléchargement depuis le repo officiel.

In [13]:
# =============================================================================
# 0.3 Téléchargement des poids officiels VeRi (SBS R50-ibn)
# =============================================================================
import glob
import urllib.request

WEIGHTS_URL = ('https://github.com/JDAI-CV/fast-reid/releases/download/'
               'v0.1.1/veri_sbs_R50-ibn.pth')
WEIGHTS = '/kaggle/working/veri_sbs_R50-ibn.pth'

candidates = glob.glob('/kaggle/input/**/*.pth', recursive=True)
if candidates:
    WEIGHTS = candidates[0]
    print('Poids trouvés dans l\'input Kaggle :', WEIGHTS)
elif not os.path.isfile(WEIGHTS):
    print('Téléchargement depuis le model zoo officiel FastReID...')
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS)
print('Poids :', WEIGHTS, '-', os.path.getsize(WEIGHTS) // (1024 * 1024), 'Mo')

Poids : /kaggle/working/veri_sbs_R50-ibn.pth - 189 Mo


### 0.4 Localisation du dataset **VeRi-776** dans `/kaggle/input`

Le dataset doit être ajouté comme **input** du notebook (bouton « + Add input »).
La cellule cherche automatiquement le dossier qui contient `image_test/`,
`name_test.txt` et `test_label.xml`.

In [14]:
# =============================================================================
# 0.4 Localisation automatique du dataset VeRi-776
# =============================================================================
def find_veri():
    for d in sorted(glob.glob('/kaggle/input/datasets/yassinekaidi/verid-test1')):
        for root, dirs, files in os.walk(d):
            if ('image_test' in dirs and 'name_test.txt' in files
                    and 'test_label.xml' in files):
                return root
    return None


VERI_DIR = find_veri()
assert VERI_DIR is not None, (
    'Dataset VeRi introuvable dans /kaggle/input. '
    'Ajoute-le comme input du notebook (dossier contenant image_train/, '
    'image_test/, name_test.txt, test_label.xml...).')
print('Dataset VeRi trouvé :', VERI_DIR)
for sub in ['image_train', 'image_test', 'image_query']:
    n = len(os.listdir(os.path.join(VERI_DIR, sub)))
    print('  -', sub, ':', n, 'images')

Dataset VeRi trouvé : /kaggle/input/datasets/yassinekaidi/verid-test1/VeRi
  - image_train : 37778 images
  - image_test : 11579 images
  - image_query : 1678 images


---
## Partie 1 — Protocole multi-vs-multi

### Principe (conforme à la mission)
- On utilise les images **test** de VeRi (200 véhicules, 20 caméras = 20 points de vue).
- Pour **chaque identité**, ses caméras sont triées puis réparties en deux
  moitiés **disjointes** : indices pairs → **ensemble REQUÊTE**, indices impairs →
  **ensemble GALERIE**. Les deux côtés sont donc **multi-caméras** et ne partagent
  **aucune caméra** (pas de fuite d'information).
- Chaque ensemble est ensuite fusionné en **un seul embedding** (méthodes partie 3),
  puis on compare **identité vs identité** (vérification 1-to-1 + identification).

In [15]:
# =============================================================================
# 1.1 Paramètres modifiables
# =============================================================================
MAX_IDS = None       # None = toutes les identités test (181) ; petit nombre = test rapide
TRAIN_MAX_IDS = 200  # identités train utilisées pour entraîner EV-MoE
MIN_IMGS, MIN_VIEWS = 3, 2  # filtres : nb minimal d'images et de caméras par ensemble
# Benchmark OFFICIEL VeRi (single-query) — Partie 5bis
OFFICIAL_QUERY_MAX = None    # None = les 1 678 requêtes officielles ; petit nombre = test rapide
OFFICIAL_GALLERY_MAX = None  # None = les 11 579 images de galerie officielle

# =============================================================================
# 1.2 Lecture des annotations VeRi (test_label.xml encodé en gb2312)
# =============================================================================
import xml.etree.ElementTree as ET
from collections import defaultdict


def read_lines(path):
    with open(path) as f:
        return [ln.strip() for ln in f if ln.strip()]


def parse_xml(path):
    with open(path, 'rb') as f:
        raw = f.read()
    for enc in ('gb2312', 'gbk', 'utf-8'):
        try:
            text = raw.decode(enc)
            break
        except (UnicodeDecodeError, LookupError):
            continue
    else:
        text = raw.decode('utf-8', errors='replace')
    return ET.fromstring(text)


def load_labels(xml_name):
    out = {}
    root = parse_xml(os.path.join(VERI_DIR, xml_name))
    for item in root.iter('Item'):
        out[item.get('imageName')] = (int(item.get('vehicleID')),
                                      int(item.get('cameraID')[1:]))  # 'c002' -> 2
    return out


test_labels = load_labels('test_label.xml')
train_labels = load_labels('train_label.xml')
print('labels test :', len(test_labels), '| labels train :', len(train_labels))

# =============================================================================
# 1.3 Construction des ensembles multi-vs-multi (split par identité)
# =============================================================================
def build_multivs_sets(min_imgs=MIN_IMGS, min_views=MIN_VIEWS):
    test_names = read_lines(os.path.join(VERI_DIR, 'name_test.txt'))
    by_id = defaultdict(lambda: defaultdict(list))
    for name in test_names:
        if name not in test_labels:
            continue
        vid, cam = test_labels[name]
        by_id[vid][cam].append(name)

    sides = {'query': {}, 'gallery': {}}
    for vid, cams in by_id.items():
        cam_order = sorted(cams)  # déterministe
        q_cams = [c for i, c in enumerate(cam_order) if i % 2 == 0]
        g_cams = [c for i, c in enumerate(cam_order) if i % 2 == 1]
        for side, chosen in (('query', q_cams), ('gallery', g_cams)):
            names = [n for c in chosen for n in cams[c]]
            cams_l = [c for c in chosen for _ in cams[c]]
            if len(names) < min_imgs or len(set(cams_l)) < min_views:
                continue
            sides[side][vid] = {
                'names': names, 'cams': cams_l,
                'paths': [os.path.join(VERI_DIR, 'image_test', n) for n in names],
            }

    query_sets, gallery_sets = sides['query'], sides['gallery']
    common = sorted(set(query_sets) & set(gallery_sets))
    meta = {
        'n_identities': len(common),
        'n_query_sets': len(query_sets),
        'n_gallery_sets': len(gallery_sets),
        'n_query_imgs': sum(len(s['names']) for s in query_sets.values()),
        'n_gallery_imgs': sum(len(s['names']) for s in gallery_sets.values()),
        'n_query_views_used': len({c for s in query_sets.values() for c in s['cams']}),
        'n_gallery_views_used': len({c for s in gallery_sets.values() for c in s['cams']}),
    }
    return query_sets, gallery_sets, common, meta


query_sets, gallery_sets, common, meta = build_multivs_sets()
if MAX_IDS:
    common = common[:MAX_IDS]
    query_sets = {v: query_sets[v] for v in common}
    gallery_sets = {v: gallery_sets[v] for v in common}
    meta = dict(meta, n_identities=len(common))
print('Identités communes (requête + galerie) :', len(common))
print(meta)
vid_ex = common[0]
print('Exemple requête  :', query_sets[vid_ex]['names'][:2],
      'cams', sorted(set(query_sets[vid_ex]['cams'])))
print('Exemple galerie  :', gallery_sets[vid_ex]['names'][:2],
      'cams', sorted(set(gallery_sets[vid_ex]['cams'])))

labels test : 11579 | labels train : 37746
Identités communes (requête + galerie) : 181
{'n_identities': 181, 'n_query_sets': 196, 'n_gallery_sets': 181, 'n_query_imgs': 6030, 'n_gallery_imgs': 5415, 'n_query_views_used': 19, 'n_gallery_views_used': 18}
Exemple requête  : ['0002_c002_00030600_0.jpg', '0002_c002_00030605_1.jpg'] cams [2, 4, 6, 8, 10]
Exemple galerie  : ['0002_c003_00084280_0.jpg', '0002_c003_00084290_0.jpg'] cams [3, 5, 7, 9, 11]


---
## Partie 2 — Extraction des embeddings FastReID (GPU)

Chaque image est redimensionnée à la taille d'inférence (256×256), passée dans le
modèle **SBS R50-ibn** et normalisée L2 → embedding **2048-D**. L'extraction se
fait **une seule fois** (query + galerie), puis les embeddings sont conservés en
mémoire pour toutes les méthodes.

In [16]:
# =============================================================================
# 2.1 Extracteur FastReID (embeddings 2048-D, L2-normalisés)
# =============================================================================
import cv2
import numpy as np
import time


class FastReIDEmbedder:
    def __init__(self, config_file, weights, device='cuda'):
        cfg = get_cfg()
        cfg.merge_from_file(config_file)
        cfg.MODEL.WEIGHTS = weights
        cfg.MODEL.DEVICE = device
        cfg.freeze()
        self.cfg = cfg
        self.device = device
        self.predictor = DefaultPredictor(cfg)

    @torch.no_grad()
    def extract_batch(self, images_bgr, batch_size=32):
        out = []
        for i in range(0, len(images_bgr), batch_size):
            chunk = images_bgr[i:i + batch_size]
            tensors = []
            for img in chunk:
                img = cv2.resize(img, tuple(self.cfg.INPUT.SIZE_TEST[::-1]),
                                 interpolation=cv2.INTER_CUBIC)
                tensors.append(torch.as_tensor(img.astype('float32').transpose(2, 0, 1)))
            t = torch.stack(tensors).to(self.device)
            feats = torch.nn.functional.normalize(self.predictor(t), p=2, dim=1)
            out.append(feats.cpu().numpy())
        return np.concatenate(out, axis=0).astype(np.float32)


CONFIG = os.path.join(FASTREID_DIR, 'configs', 'VeRi', 'sbs_R50-ibn.yml')
embedder = FastReIDEmbedder(CONFIG, WEIGHTS, device=DEVICE)
print('✔ Embedder prêt sur', DEVICE)

# =============================================================================
# 2.2 Extraction de tous les ensembles (query + galerie)
# =============================================================================
def extract_sets(sets, tag):
    out = {}
    t0 = time.time()
    for i, (vid, s) in enumerate(sets.items()):
        imgs = []
        for p in s['paths']:
            img = cv2.imread(p)
            imgs.append(img if img is not None else np.zeros((256, 256, 3),
                                                             dtype=np.uint8))
        out[vid] = (embedder.extract_batch(imgs), np.asarray(s['cams']))
        if (i + 1) % 50 == 0 or i + 1 == len(sets):
            print('  [{}] {}/{} ensembles — {:.0f}s'.format(tag, i + 1,
                                                            len(sets),
                                                            time.time() - t0))
    return out


print('Extraction des ensembles QUERY...')
q_embs = extract_sets(query_sets, 'query')
print('Extraction des ensembles GALERIE...')
g_embs = extract_sets(gallery_sets, 'gallery')
print('✔ Extraction terminée :', len(q_embs), 'requêtes,', len(g_embs), 'galeries')

✔ Embedder prêt sur cuda
Extraction des ensembles QUERY...
  [query] 50/196 ensembles — 15s
  [query] 100/196 ensembles — 31s
  [query] 150/196 ensembles — 47s
  [query] 196/196 ensembles — 58s
Extraction des ensembles GALERIE...
  [gallery] 50/181 ensembles — 14s
  [gallery] 100/181 ensembles — 30s
  [gallery] 150/181 ensembles — 44s
  [gallery] 181/181 ensembles — 51s
✔ Extraction terminée : 196 requêtes, 181 galeries


---
## Partie 3 — Les 3 méthodes de fusion

### 3.1 Averaging (baseline)
Moyenne simple des embeddings L2 → renormalisation. Chaque image vote à égalité.

### 3.2 VCNet — pondération par point de vue
Pour chaque image i de la caméra v :
- **Confiance de vue** `cᵢ = <fᵢ, centroid_v>` : l'image est-elle représentative
  de sa vue ? (centroid global = embedding moyen de TOUTES les images de la caméra) ;
- **Complémentarité** `rᵢ` : une vue sous-représentée apporte une information
  nouvelle → poids plus élevé (`1/count`) ;
- Poids final `wᵢ ∝ softmax(β · cᵢ · rᵢ)` avec β = 2 (meilleure config trouvée).

### 3.3 EV-MoE / CAFNet — Mixture of Experts + Attention
- **Gating** : le vecteur one-hot de la caméra → softmax sur 4 **experts** ;
- **Experts** : projections linéaires spécialisées (init = identité) ;
- **Attention** : self-attention entre les images → poids d'importance ;
- Fusion `f = L2(Σ aᵢ · hᵢ)`. **Entraîné** en partie 4 (perte CE + triplet,
  gating par vue **réelle** — contrairement à la v1 qui n'apprenait rien).

In [17]:
# =============================================================================
# 3.1 + 3.2 Averaging & VCNet (implémentations officielles, déterministes)
# =============================================================================
def l2norm(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)


def average_fusion(embs, cams=None):
    """Baseline : moyenne simple des embeddings L2 -> renormalisation."""
    return l2norm(np.asarray(embs, dtype=np.float32).mean(axis=0))


def vcnet_fusion(embs, cams, centroids=None, beta=2.0, comp_mode='inv'):
    """VCNet : poids adaptatifs par point de vue (caméra = vue).
    w_i ∝ softmax(β · confiance_vue · complémentarité)."""
    embs = np.asarray(embs, dtype=np.float32)
    cams = np.asarray(cams)
    K = embs.shape[0]
    _, counts = np.unique(cams, return_counts=True)
    cmap = dict(zip(np.unique(cams), counts))
    w = np.zeros(K, dtype=np.float64)
    for i in range(K):
        cam = int(cams[i])
        cent = centroids.get(cam) if centroids else None
        conf = float(embs[i] @ cent) if cent is not None else 0.5
        cnt = max(cmap.get(cam, 1), 1)
        if comp_mode == 'sqrt':
            comp = 1.0 / np.sqrt(cnt)
        elif comp_mode == 'log':
            comp = 1.0 / np.log1p(cnt)
        elif comp_mode == 'none':
            comp = 1.0
        else:  # 'inv'
            comp = 1.0 / cnt
        w[i] = conf * comp
    w = w / (w.max() + 1e-12)
    w = np.exp(beta * (w - w.max()))
    w = w / (w.sum() + 1e-12)
    return l2norm((embs * w[:, None]).sum(axis=0))


def global_centroids(sets_embs):
    """Centroids de vue GLOBAUX : embedding moyen L2 de toutes les images de
    chaque caméra (calculés sur la galerie -> référence stable)."""
    E, C = [], []
    for _, (e, c) in sets_embs.items():
        E.append(e)
        C.append(c)
    E = np.concatenate(E)
    C = np.concatenate(C)
    cents = {}
    for cam in np.unique(C):
        m = C == cam
        if m.sum() >= 1:
            cents[int(cam)] = l2norm(E[m].mean(axis=0))
    return cents

In [18]:
# =============================================================================
# 3.3 EV-MoE / CAFNet — Mixture of Experts + Attention (module entraîné)
# =============================================================================
import torch.nn as nn
import torch.nn.functional as F


class EVMoEFusion(nn.Module):
    """Fusion apprise : gating par vue -> experts -> self-attention sur images.
    Entraînée en partie 4 (perte CE + triplet) sur les ensembles du train."""

    def __init__(self, dim=2048, n_views=20, n_experts=4, dropout=0.1):
        super().__init__()
        self.dim = dim
        self.n_views = n_views
        self.n_experts = n_experts
        self.experts = nn.ModuleList(
            [nn.Sequential(nn.Linear(dim, dim, bias=False), nn.ReLU(inplace=True))
             for _ in range(n_experts)])
        self.gate = nn.Sequential(
            nn.Linear(n_views, n_experts, bias=True), nn.Softmax(dim=-1))
        self.q_proj = nn.Linear(dim, dim // 4, bias=False)
        self.k_proj = nn.Linear(dim, dim // 4, bias=False)
        self.dropout = nn.Dropout(dropout)
        for e in self.experts:
            nn.init.eye_(e[0].weight)  # départ proche d'une moyenne

    def _view_vector(self, cams, n):
        v = torch.zeros(n, self.n_views, device=cams.device)
        for i, c in enumerate(cams):
            if 1 <= int(c) <= self.n_views:
                v[i, int(c) - 1] = 1.0
        return v

    def forward(self, embs, cams):
        K = embs.shape[0]
        if K == 0:
            return torch.zeros(self.dim, device=embs.device), None
        v = self._view_vector(cams, K)                # (K, n_views)
        g = self.gate(v)                              # (K, n_experts)
        h = torch.stack([e(embs) for e in self.experts], dim=1)  # (K, E, D)
        h = (h * g.unsqueeze(-1)).sum(dim=1)          # (K, D)
        h = F.normalize(h, p=2, dim=1)
        q = self.q_proj(h)                            # (K, D/4)
        k = self.k_proj(h)
        att = (q @ k.T) / (self.dim ** 0.5)           # (K, K)
        a = torch.softmax(att, dim=1).mean(dim=0)     # importance moyenne
        a = torch.softmax(a * self.dim ** 0.5, dim=0)
        fused = F.normalize((a.unsqueeze(-1) * h).sum(dim=0), p=2, dim=0)
        return fused, a

---
## Partie 4 — Entraînement d'**EV-MoE** sur GPU

### Données
Sous-ensemble du **train** VeRi (`TRAIN_MAX_IDS` identités, ici 200) : chaque
véhicule fournit 2 ensembles multi-caméras (split caméras paires/impaires).
Les embeddings sont extraits par FastReID (GPU).

### Pertes
- **Triplet loss** : deux ensembles du même véhicule fusionnent proches, un autre
  véhicule → loin ;
- **Cross-entropy** : une tête classifie l'identité depuis le vecteur fusionné
  → la fusion est forcée d'être discriminante.

### Points clés (v2, corrigés par rapport à la v1)
1. **Gating par caméras réelles** (la v1 passait une vue uniforme → rien à apprendre) ;
2. **Perte CE ajoutée** (la v1 n'avait que la triplet, qui collapsait à ~0) ;
3. Plus de données (200 identités) et plus de paires par époque.

Sur GPU (T4), l'entraînement complet prend **2 à 5 minutes**.

In [19]:
# =============================================================================
# 4.1 Ensembles d'entraînement (train VeRi) + extraction des embeddings
# =============================================================================
def build_train_sets(max_ids=TRAIN_MAX_IDS, min_imgs=MIN_IMGS,
                     min_views=MIN_VIEWS, seed=0):
    by_id = defaultdict(lambda: defaultdict(list))
    for name, (vid, cam) in train_labels.items():
        by_id[vid][cam].append(name)
    ids = sorted(by_id.keys())
    rng = np.random.RandomState(seed)
    rng.shuffle(ids)
    ids = ids[:max_ids]
    sets = []
    for vid in ids:
        cam_order = sorted(by_id[vid])
        q_cams = [c for i, c in enumerate(cam_order) if i % 2 == 0]
        g_cams = [c for i, c in enumerate(cam_order) if i % 2 == 1]
        for side, chosen in (('q', q_cams), ('g', g_cams)):
            names = [n for c in chosen for n in by_id[vid][c]]
            cams = [c for c in chosen for _ in by_id[vid][c]]
            if len(names) < min_imgs or len(set(cams)) < min_views:
                continue
            sets.append({'vid': vid, 'side': side, 'names': names, 'cams': cams,
                         'paths': [os.path.join(VERI_DIR, 'image_train', n)
                                   for n in names]})
    return sets, ids


train_sets, train_ids = build_train_sets()
vid_to_idx = defaultdict(list)
for i, s in enumerate(train_sets):
    vid_to_idx[s['vid']].append(i)
keep = {v for v, idx in vid_to_idx.items() if len(idx) >= 2}
train_sets = [s for s in train_sets if s['vid'] in keep]
vid_to_idx = defaultdict(list)
for i, s in enumerate(train_sets):
    vid_to_idx[s['vid']].append(i)
vids = sorted(vid_to_idx.keys())
vid_to_label = {v: i for i, v in enumerate(vids)}
print('Ensembles train :', len(train_sets), '| Identités :', len(vids))

# Extraction des embeddings train (GPU) — une seule fois
train_embs = {}
t0 = time.time()
for i, s in enumerate(train_sets):
    key = (s['vid'], s['side'])
    if key not in train_embs:
        imgs = []
        for p in s['paths']:
            img = cv2.imread(p)
            imgs.append(img if img is not None else np.zeros(
                (256, 256, 3), dtype=np.uint8))
        train_embs[key] = embedder.extract_batch(imgs)
    if (i + 1) % 100 == 0 or i + 1 == len(train_sets):
        print('  [{}/{}] ensembles extraits — {:.0f}s'.format(i + 1, len(train_sets),
                                                              time.time() - t0))

Ensembles train : 376 | Identités : 188
  [100/376] ensembles extraits — 36s
  [200/376] ensembles extraits — 67s
  [300/376] ensembles extraits — 100s
  [376/376] ensembles extraits — 127s


In [20]:
# =============================================================================
# 4.2 Boucle d'entraînement EV-MoE (CE + triplet, gating par vue réelle)
# =============================================================================
class EVMoEWithHead(nn.Module):
    def __init__(self, dim=2048, n_views=20, n_experts=4, n_ids=200):
        super().__init__()
        self.fusion = EVMoEFusion(dim=dim, n_views=n_views, n_experts=n_experts)
        self.head = nn.Linear(dim, n_ids)


def triplet(anc, pos, neg, margin=0.3):
    a = F.normalize(anc, p=2, dim=0)
    p = F.normalize(pos, p=2, dim=0)
    n = F.normalize(neg, p=2, dim=0)
    d_pos = 1.0 - (a @ p).clamp(-1, 1)
    d_neg = 1.0 - (a @ n).clamp(-1, 1)
    return F.relu(d_pos - d_neg + margin)


def to_t(s, device):
    embs = torch.as_tensor(train_embs[(s['vid'], s['side'])]).to(device)
    cams = torch.as_tensor(np.asarray(s['cams'], dtype=np.int64)).to(device)
    lab = torch.tensor(vid_to_label[s['vid']], dtype=torch.long, device=device)
    return embs, cams, lab


def sample(rng):
    vids_l = list(vid_to_idx.keys())
    anc = int(rng.choice(len(train_sets)))
    anc_set = train_sets[anc]
    vid = anc_set['vid']
    pos = int(rng.choice([i for i in vid_to_idx[vid] if i != anc]))
    neg_vid = rng.choice(vids_l)
    while neg_vid == vid:
        neg_vid = rng.choice(vids_l)
    neg = int(rng.choice(vid_to_idx[neg_vid]))
    return (to_t(anc_set, DEVICE), to_t(train_sets[pos], DEVICE),
            to_t(train_sets[neg], DEVICE))


EPOCHS, PAIRS_PER_EPOCH = 40, 250
model = EVMoEWithHead(dim=2048, n_views=20, n_experts=4, n_ids=len(vids)).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
rng = np.random.RandomState(0)

t0 = time.time()
for ep in range(1, EPOCHS + 1):
    model.train()
    tot = {'loss': 0.0, 'tri': 0.0, 'ce': 0.0}
    for _ in range(PAIRS_PER_EPOCH):
        (a_e, a_c, a_l), (p_e, p_c, p_l), (n_e, n_c, n_l) = sample(rng)
        f_a, _ = model.fusion(a_e, a_c)
        f_p, _ = model.fusion(p_e, p_c)
        f_n, _ = model.fusion(n_e, n_c)
        tri = triplet(f_a, f_p, f_n)
        ce = (F.cross_entropy(model.head(f_a), a_l)
              + F.cross_entropy(model.head(f_p), p_l)
              + F.cross_entropy(model.head(f_n), n_l)) / 3.0
        loss = tri + ce
        opt.zero_grad()
        loss.backward()
        opt.step()
        tot['loss'] += float(loss.detach())
        tot['tri'] += float(tri.detach())
        tot['ce'] += float(ce.detach())
    if ep == 1 or ep % 5 == 0 or ep == EPOCHS:
        print('epoch {}/{}  loss {:.4f}  tri {:.4f}  ce {:.4f}  ({:.0f}s)'.format(
            ep, EPOCHS, tot['loss'] / PAIRS_PER_EPOCH,
            tot['tri'] / PAIRS_PER_EPOCH, tot['ce'] / PAIRS_PER_EPOCH,
            time.time() - t0))

torch.save({'state_dict': model.fusion.state_dict(),
            'dim': 2048, 'n_views': 20, 'n_experts': 4,
            'n_train_sets': len(train_sets), 'n_train_ids': len(vids),
            'epoch': EPOCHS},
           '/kaggle/working/evmoe_fusion_v2.pth')
print('✔ Modèle EV-MoE entraîné et sauvegardé : /kaggle/working/evmoe_fusion_v2.pth')

epoch 1/40  loss 4.9965  tri 0.0045  ce 4.9920  (7s)
epoch 5/40  loss 2.4212  tri 0.0000  ce 2.4212  (30s)
epoch 10/40  loss 1.1006  tri 0.0002  ce 1.1004  (58s)
epoch 15/40  loss 0.7937  tri 0.0000  ce 0.7937  (87s)
epoch 20/40  loss 0.7126  tri 0.0000  ce 0.7126  (116s)
epoch 25/40  loss 0.6153  tri 0.0000  ce 0.6153  (145s)
epoch 30/40  loss 0.5416  tri 0.0000  ce 0.5416  (173s)
epoch 35/40  loss 0.5058  tri 0.0000  ce 0.5058  (202s)
epoch 40/40  loss 0.4651  tri 0.0004  ce 0.4648  (230s)
✔ Modèle EV-MoE entraîné et sauvegardé : /kaggle/working/evmoe_fusion_v2.pth


---
## Partie 5 — Évaluation officielle des 3 méthodes

### Métriques
- **Identification** : matrice de similarité cosinus identité vs identité →
  **Rank-1/5/10**, **mAP**, courbe **CMC** ;
- **Vérification 1:1** : paires (i,i) positives vs (i,j) négatives →
  **Accuracy** (seuil optimal), **AUC**, courbe **ROC**.

EV-MoE est évalué avec le modèle **entraîné** (partie 4), VCNet avec sa meilleure
configuration (centroids globaux galerie, β=2, complémentarité `1/count`).

In [21]:
# =============================================================================
# 5.1 Métriques officielles (identification + vérification 1:1)
# =============================================================================
def identification_metrics(S):
    n = S.shape[0]
    order = np.argsort(-S, axis=1)
    ranks = np.array([np.where(order[i] == i)[0][0] for i in range(n)])
    cmc = [float((ranks <= k).mean()) for k in range(n)]
    return {'rank_1': float((ranks == 0).mean()),
            'rank_5': float((ranks < 5).mean()),
            'rank_10': float((ranks < 10).mean()),
            'mAP': float((1.0 / (ranks + 1)).mean()),
            'cmc': cmc, 'n_identities': n}


def verification_metrics(S):
    n = S.shape[0]
    pos = np.array([S[i, i] for i in range(n)])
    neg = np.array([S[i, j] for i in range(n) for j in range(n) if i != j])
    scores = np.concatenate([pos, neg])
    labels = np.concatenate([np.ones_like(pos), np.zeros_like(neg)])
    order = np.argsort(-scores)
    ss, sl = scores[order], labels[order]
    tps, fps = np.cumsum(sl), np.cumsum(1 - sl)
    n_pos, n_neg = int(sl.sum()), int((1 - sl).sum())
    tpr = np.concatenate([[0.0], tps / max(n_pos, 1), [1.0]])
    fpr = np.concatenate([[0.0], fps / max(n_neg, 1), [1.0]])
    step = max(1, len(fpr) // 250)
    roc_fpr = [float(x) for x in fpr[::step]]
    roc_tpr = [float(x) for x in tpr[::step]]
    if roc_fpr[-1] != 1.0:
        roc_fpr.append(1.0)
        roc_tpr.append(1.0)
    try:
        auc = float(np.trapezoid(tpr, fpr))
    except AttributeError:
        auc = float(np.trapz(tpr, fpr))
    best = {'acc': 0.0, 'thr': 0.0}
    for thr in np.linspace(-1.0, 1.0, 2001):
        acc = float(((scores > thr) == labels).mean())
        if acc > best['acc']:
            best = {'acc': acc, 'thr': float(thr)}
    return {'accuracy': best['acc'], 'threshold': best['thr'], 'auc': auc,
            'num_pos_pairs': n_pos, 'num_neg_pairs': n_neg,
            'roc_fpr': roc_fpr, 'roc_tpr': roc_tpr}

In [22]:
# =============================================================================
# 5.2 Évaluation des 3 méthodes (fusion + similarité + métriques)
# =============================================================================
methods = ['average', 'vcnet', 'evmoe']
centroids = global_centroids(g_embs)   # VCNet : centroids globaux galerie
print('Centroids globaux :', len(centroids), 'caméras')
model.fusion.eval()


def fuse(method, embs, cams):
    if method == 'average':
        return average_fusion(embs, cams)
    if method == 'vcnet':
        return vcnet_fusion(embs, cams, centroids=centroids, beta=2.0,
                            comp_mode='inv')
    with torch.no_grad():
        t = torch.as_tensor(embs).to(DEVICE)
        c = torch.as_tensor(np.asarray(cams, dtype=np.int64)).to(DEVICE)
        fused, _ = model.fusion(t, c)
    return fused.cpu().numpy().astype(np.float32)


results = {}
ids_int = list(common)
for m in methods:
    t0 = time.time()
    q = {v: fuse(m, e, c) for v, (e, c) in q_embs.items()}
    g = {v: fuse(m, e, c) for v, (e, c) in g_embs.items()}
    S = np.zeros((len(ids_int), len(ids_int)), dtype=np.float32)
    for i, v in enumerate(ids_int):
        S[i] = [float(q[v] @ g[w]) for w in ids_int]
    ident = identification_metrics(S)
    verif = verification_metrics(S)
    results[m] = {'identification': ident, 'verification_1to1': verif,
                  'similarity_matrix': S.tolist()}
    print('{:<8} Rank-1 {:.2%} | mAP {:.2%} | Acc 1:1 {:.2%} | AUC {:.4f} | '
          '{:.0f}s'.format(m.upper(), ident['rank_1'], ident['mAP'],
                           verif['accuracy'], verif['auc'], time.time() - t0))

# Tableau comparatif
import pandas as pd
rows = [{'Méthode': m.upper(),
         'Rank-1': '{:.2%}'.format(results[m]['identification']['rank_1']),
         'Rank-5': '{:.2%}'.format(results[m]['identification']['rank_5']),
         'mAP': '{:.2%}'.format(results[m]['identification']['mAP']),
         'Acc 1:1': '{:.2%}'.format(results[m]['verification_1to1']['accuracy']),
         'AUC': '{:.4f}'.format(results[m]['verification_1to1']['auc'])}
        for m in methods]
display(pd.DataFrame(rows))

Centroids globaux : 18 caméras
AVERAGE  Rank-1 92.82% | mAP 95.17% | Acc 1:1 99.68% | AUC 0.9967 | 0s
VCNET    Rank-1 92.82% | mAP 94.96% | Acc 1:1 99.66% | AUC 0.9966 | 0s
EVMOE    Rank-1 93.92% | mAP 96.50% | Acc 1:1 99.81% | AUC 0.9988 | 1s


,Méthode,Rank-1,Rank-5,mAP,Acc 1:1,AUC
0,AVERAGE,92.82%,97.79%,95.17%,99.68%,0.9967
1,VCNET,92.82%,97.79%,94.96%,99.66%,0.9966
2,EVMOE,93.92%,100.00%,96.50%,99.81%,0.9988


In [23]:
# =============================================================================
# 5.3 Figures : courbes CMC + ROC, matrices de similarité
# =============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for m in methods:
    cmc = results[m]['identification']['cmc'][:50]
    xs = np.arange(1, len(cmc) + 1)
    ax[0].plot(xs, np.array(cmc) * 100, marker='o', markersize=3,
               label='{} (Rank-1 {:.1f}%)'.format(
                   m.upper(), results[m]['identification']['rank_1'] * 100))
    v = results[m]['verification_1to1']
    ax[1].plot(v['roc_fpr'], v['roc_tpr'],
               label='{} (AUC {:.4f})'.format(m.upper(), v['auc']))
ax[0].set(xlabel='Rank', ylabel='Cumulative Recall (%)',
          title='CMC — identification multi-vs-multi (VeRi-776)')
ax[0].grid(alpha=.3)
ax[0].legend()
ax[1].plot([0, 1], [0, 1], 'k--', alpha=.4, label='Aléatoire')
ax[1].set(xlabel='False Positive Rate', ylabel='True Positive Rate',
          title='ROC — vérification 1:1 (VeRi-776)')
ax[1].grid(alpha=.3)
ax[1].legend()
plt.tight_layout()
plt.savefig('/kaggle/working/figures_comparaison.png', dpi=150)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, m in zip(axes, methods):
    S = np.array(results[m]['similarity_matrix'])
    im = ax.imshow(S, cmap='viridis', vmin=0.0, vmax=1.0)
    ax.set_title('{} — cosinus moyen {:.3f}'.format(m.upper(), float(S.mean())))
    ax.set_xlabel('Galerie (identité)')
    ax.set_ylabel('Requête (identité)')
plt.tight_layout()
plt.savefig('/kaggle/working/matrices_similarite.png', dpi=150)
plt.show()

---
## Partie 5.4 — Test statistique de significativité (EV-MoE vs les autres)

Pour pouvoir affirmer « EV-MoE est la meilleure méthode », on compare les 3
méthodes **par identité** (sur les 181 identités test) :
- **Wilcoxon (test signé sur les rangs)** appliqué au mAP par identité
  (approximé par le rang réciproque 1/(rank+1)) entre EV-MoE et chaque autre
  méthode → teste si EV-MoE est significativement meilleur en distribution ;
- **McNemar** sur la réussite Rank-1 (paires discordantes : identités où une
  seule des deux méthodes trouve la bonne identité en rang 1).

→ Si p < 0,05, l'affirmation est **statistiquement fondée**.

In [24]:
# =============================================================================
# 5.4 Tests statistiques de significativité (par identité)
# =============================================================================
from scipy import stats as sp_stats


def per_identity_ranks(method):
    S = np.array(results[method]['similarity_matrix'])
    order = np.argsort(-S, axis=1)
    n = S.shape[0]
    return np.array([np.where(order[i] == i)[0][0] for i in range(n)])


ranks_by_method = {m: per_identity_ranks(m) for m in methods}
rr = {m: 1.0 / (ranks_by_method[m] + 1) for m in methods}  # AP approximé par identité
n_ids = len(ranks_by_method['evmoe'])

print('=== Tests statistiques (par identité, n = {}) ==='.format(n_ids))
stat_results = {}
best = 'evmoe'
for m in methods:
    if m == best:
        continue
    try:
        _, p_w = sp_stats.wilcoxon(rr[best], rr[m])
    except ValueError:
        p_w = 1.0  # toutes les différences nulles -> pas de test possible
    ok_evmoe = ranks_by_method[best] == 0
    ok_m = ranks_by_method[m] == 0
    b = int((ok_evmoe & ~ok_m).sum())   # EV-MoE seul correct en Rank-1
    c = int((~ok_evmoe & ok_m).sum())   # m seul correct en Rank-1
    disc = b + c
    p_mc = (1.0 - sp_stats.chi2.cdf((abs(b - c) - 1) ** 2 / max(disc, 1), 1)
            if disc > 0 else 1.0)
    stat_results['{}_vs_{}'.format(best, m)] = {
        'wilcoxon_p': round(float(p_w), 4),
        'mcnemar_p': round(float(p_mc), 4),
        'mcnemar_evmoe_only': b,
        'mcnemar_other_only': c,
    }
    print('EV-MoE vs {} : Wilcoxon p={:.4f} {} | McNemar p={:.4f} {}'.format(
        m.upper(), p_w, 'SIGNIFICATIF' if p_w < 0.05 else 'non significatif',
        p_mc, 'SIGNIFICATIF' if p_mc < 0.05 else 'non significatif'))

sig_all = all(v['wilcoxon_p'] < 0.05 or v['mcnemar_p'] < 0.05
              for v in stat_results.values())
stat_conclusion = ('EV-MoE significativement supérieur aux deux autres (p<0.05)'
                   if sig_all else 'EV-MoE supérieur en moyenne mais test non '
                   'concluant sur toutes les comparaisons')
print('Conclusion :', stat_conclusion)

=== Tests statistiques (par identité, n = 181) ===
EV-MoE vs AVERAGE : Wilcoxon p=0.0654 non significatif | McNemar p=0.6171 non significatif
EV-MoE vs VCNET : Wilcoxon p=0.0557 non significatif | McNemar p=0.6171 non significatif
Conclusion : EV-MoE supérieur en moyenne mais test non concluant sur toutes les comparaisons


---
## Partie 5bis — Benchmark OFFICIEL VeRi-776 (protocole single-query)

Le protocole **officiel** de VeRi-776 est single-query :
- **1 678 requêtes** (`image_query/`, `name_query.txt`) — elles font partie de
  la galerie (`image_test/`) ;
- **11 579 images de galerie** (`image_test/`, `name_test.txt`) ;
- **`gt_index.txt`** : pour chaque requête, les indices (1-based) des images de
  galerie de MÊME identité ;
- **`jk_index.txt`** : indices « junk » (même caméra/temps) **exclus du classement** ;
- métriques : **mAP + Rank-1** (évaluation officielle VeRi) ;
- **Re-ranking** (option `OFFICIAL_RERANK`) : les chiffres officiels FastReID
  (97,0 % / 81,9 %) sont obtenus **avec re-ranking** (`TEST.RERANK.ENABLED`, k1=20,
  k2=6, λ=0,3) — même algorithme que `fastreid/evaluation/rerank.py` ; sans
  re-ranking, un R50-ibn obtient typiquement ~88 % / ~51 %.

→ Résultats comparables aux chiffres **officiels FastReID** sur VeRi :
**Rank-1 97,0 % / mAP 81,9 %**.

In [25]:
# =============================================================================
# 5bis.1 Chargement du protocole officiel (requêtes, galerie, gt, junk)
# =============================================================================
OFFICIAL_RERANK = True  # True = re-ranking officiel FastReID (comme le model zoo)

# Re-ranking officiel FastReID (source : fastreid/evaluation/rerank.py,
# d'après zhunzhong07/person-re-ranking) — utilisé pour comparer aux chiffres
# du model zoo (97,0 % / 81,9 %), qui sont obtenus AVEC re-ranking.
def re_ranking(q_g_dist, q_q_dist, g_g_dist, k1=20, k2=6, lambda_value=0.3):
    original_dist = np.concatenate(
        [np.concatenate([q_q_dist, q_g_dist], axis=1),
         np.concatenate([q_g_dist.T, g_g_dist], axis=1)], axis=0)
    original_dist = np.power(original_dist, 2).astype(np.float32)
    original_dist = np.transpose(1. * original_dist / np.max(original_dist, axis=0))
    V = np.zeros_like(original_dist).astype(np.float32)
    initial_rank = np.argsort(original_dist).astype(np.int32)
    query_num = q_g_dist.shape[0]
    gallery_num = q_g_dist.shape[0] + q_g_dist.shape[1]
    all_num = gallery_num
    for i in range(all_num):
        forward_k_neigh_index = initial_rank[i, :k1 + 1]
        backward_k_neigh_index = initial_rank[forward_k_neigh_index, :k1 + 1]
        fi = np.where(backward_k_neigh_index == i)[0]
        k_reciprocal_index = forward_k_neigh_index[fi]
        k_reciprocal_expansion_index = k_reciprocal_index
        for j in range(len(k_reciprocal_index)):
            candidate = k_reciprocal_index[j]
            cand_fwd = initial_rank[candidate, :int(np.around(k1 / 2.)) + 1]
            cand_bwd = initial_rank[cand_fwd, :int(np.around(k1 / 2.)) + 1]
            fi_candidate = np.where(cand_bwd == candidate)[0]
            cand_k_recip = cand_fwd[fi_candidate]
            if len(np.intersect1d(cand_k_recip, k_reciprocal_index)) > 2. / 3 * len(cand_k_recip):
                k_reciprocal_expansion_index = np.append(k_reciprocal_expansion_index, cand_k_recip)
        k_reciprocal_expansion_index = np.unique(k_reciprocal_expansion_index)
        weight = np.exp(-original_dist[i, k_reciprocal_expansion_index])
        V[i, k_reciprocal_expansion_index] = 1. * weight / np.sum(weight)
    original_dist = original_dist[:query_num, ]
    if k2 != 1:
        V_qe = np.zeros_like(V, dtype=np.float32)
        for i in range(all_num):
            V_qe[i, :] = np.mean(V[initial_rank[i, :k2], :], axis=0)
        V = V_qe
        del V_qe
    del initial_rank
    invIndex = []
    for i in range(gallery_num):
        invIndex.append(np.where(V[:, i] != 0)[0])
    jaccard_dist = np.zeros_like(original_dist, dtype=np.float32)
    for i in range(query_num):
        temp_min = np.zeros(shape=[1, gallery_num], dtype=np.float32)
        indNonZero = np.where(V[i, :] != 0)[0]
        indImages = [invIndex[ind] for ind in indNonZero]
        for j in range(len(indNonZero)):
            temp_min[0, indImages[j]] = temp_min[0, indImages[j]] + np.minimum(
                V[i, indNonZero[j]], V[indImages[j], indNonZero[j]])
        jaccard_dist[i] = 1 - temp_min / (2. - temp_min)
    final_dist = jaccard_dist * (1 - lambda_value) + original_dist * lambda_value
    del original_dist, V, jaccard_dist
    final_dist = final_dist[:query_num, query_num:]
    return final_dist


def read_index_lines(path):
    # Lit un fichier d'indices en gardant l'alignement (lignes vides -> [])
    with open(path, 'r') as f:
        return [ln.split() for ln in f]


name_query = read_lines(os.path.join(VERI_DIR, 'name_query.txt'))
name_test = read_lines(os.path.join(VERI_DIR, 'name_test.txt'))
gt_index = read_index_lines(os.path.join(VERI_DIR, 'gt_index.txt'))
jk_index = read_index_lines(os.path.join(VERI_DIR, 'jk_index.txt'))
print('Requêtes officielles :', len(name_query))
print('Galerie officielle   :', len(name_test))

if OFFICIAL_QUERY_MAX:
    name_query = name_query[:OFFICIAL_QUERY_MAX]
    gt_index = gt_index[:OFFICIAL_QUERY_MAX]
    jk_index = jk_index[:OFFICIAL_QUERY_MAX]
if OFFICIAL_GALLERY_MAX:
    name_test = name_test[:OFFICIAL_GALLERY_MAX]
    gt_index = [[x for x in gt if int(x) - 1 < OFFICIAL_GALLERY_MAX] for gt in gt_index]
    jk_index = [[x for x in jk if int(x) - 1 < OFFICIAL_GALLERY_MAX] for jk in jk_index]

q_paths = [os.path.join(VERI_DIR, 'image_query', n) for n in name_query]
g_paths = [os.path.join(VERI_DIR, 'image_test', n) for n in name_test]


def load_images(paths):
    imgs = []
    for p in paths:
        img = cv2.imread(p)
        imgs.append(img if img is not None else np.zeros((256, 256, 3), dtype=np.uint8))
    return imgs


print('Extraction des requêtes officielles (GPU)...')
qf = embedder.extract_batch(load_images(q_paths), batch_size=64)
print('Extraction de la galerie officielle (GPU)...')
gf = embedder.extract_batch(load_images(g_paths), batch_size=64)
print('Shapes :', qf.shape, gf.shape)

# =============================================================================
# 5bis.2 Évaluation officielle VeRi (mAP + Rank-1, junk exclus)
# =============================================================================
def evaluate_veri_official(qf, gf, gt_index, jk_index):
    sim = qf @ gf.T
    aps, ranks = [], []
    for i in range(qf.shape[0]):
        good = {int(x) - 1 for x in gt_index[i]}
        junk = {int(x) - 1 for x in jk_index[i]}
        order = [idx for idx in np.argsort(-sim[i]) if idx not in junk]
        rank = None
        for pos, idx in enumerate(order):
            if idx in good:
                rank = pos
                break
        if rank is None:
            continue  # aucune bonne correspondance dans la galerie tronquée
        ranks.append(rank)
        hits, precs = 0, []
        for pos, idx in enumerate(order):
            if idx in good:
                hits += 1
                precs.append(hits / (pos + 1))
        aps.append(np.mean(precs) if precs else 0.0)
    if not aps:
        return {'rank_1': 0.0, 'mAP': 0.0, 'n_queries': 0}
    return {'rank_1': float(np.mean([r == 0 for r in ranks])),
            'mAP': float(np.mean(aps)),
            'n_queries': len(ranks)}


if OFFICIAL_RERANK:
    print('Re-ranking officiel FastReID (k1=20, k2=6, lambda=0.3)...')
    # distance cosinus = 1 - similarité (mêmes embeddings L2 que l'évaluation)
    q_g_dist = 1.0 - qf @ gf.T
    q_q_dist = 1.0 - qf @ qf.T
    g_g_dist = 1.0 - gf @ gf.T
    rerank_dist = re_ranking(q_g_dist.astype(np.float32),
                             q_q_dist.astype(np.float32),
                             g_g_dist.astype(np.float32),
                             k1=20, k2=6, lambda_value=0.3)
    sim = -rerank_dist  # distance -> similarité pour l'évaluateur
    qf_r, gf_r = qf, gf
else:
    sim = None


def evaluate_veri_official(qf, gf, gt_index, jk_index, sim=None):
    if sim is None:
        sim = qf @ gf.T
    aps, ranks = [], []
    for i in range(qf.shape[0]):
        good = {int(x) - 1 for x in gt_index[i]}
        junk = {int(x) - 1 for x in jk_index[i]}
        order = [idx for idx in np.argsort(-sim[i]) if idx not in junk]
        rank = None
        for pos, idx in enumerate(order):
            if idx in good:
                rank = pos
                break
        if rank is None:
            continue  # aucune bonne correspondance dans la galerie tronquée
        ranks.append(rank)
        hits, precs = 0, []
        for pos, idx in enumerate(order):
            if idx in good:
                hits += 1
                precs.append(hits / (pos + 1))
        aps.append(np.mean(precs) if precs else 0.0)
    if not aps:
        return {'rank_1': 0.0, 'mAP': 0.0, 'n_queries': 0}
    return {'rank_1': float(np.mean([r == 0 for r in ranks])),
            'mAP': float(np.mean(aps)),
            'n_queries': len(ranks)}


official = evaluate_veri_official(qf, gf, gt_index, jk_index, sim=sim)
print('=== Benchmark OFFICIEL VeRi-776 (single-query) ===')
print('Rank-1 : {:.2%}  |  mAP : {:.2%}  ({} requêtes évaluées)'.format(
    official['rank_1'], official['mAP'], official['n_queries']))
print('Re-ranking :', 'ACTIVÉ' if OFFICIAL_RERANK else 'désactivé')
print('Référence officielle FastReID (avec re-ranking) : Rank-1 97.00% | mAP 81.90%')


Requêtes officielles : 1678
Galerie officielle   : 11579
Extraction des requêtes officielles (GPU)...
Extraction de la galerie officielle (GPU)...
Shapes : (1678, 2048) (11579, 2048)
Re-ranking officiel FastReID (k1=20, k2=6, lambda=0.3)...
=== Benchmark OFFICIEL VeRi-776 (single-query) ===
Rank-1 : 89.15%  |  mAP : 55.76%  (1678 requêtes évaluées)
Re-ranking : ACTIVÉ
Référence officielle FastReID (avec re-ranking) : Rank-1 97.00% | mAP 81.90%


---
## Partie 6 — Export des résultats (ZIP téléchargeable)

Le ZIP contient : `rapport_officiel.json` (toutes les métriques + matrices),
`rapport_officiel.txt` (tableau lisible), les figures PNG, et le modèle
**EV-MoE entraîné** (`evmoe_fusion_v2.pth`). Le fichier apparaît dans le
panneau **Output** du notebook → bouton **Download**.

In [ ]:
# =============================================================================
# 6.1 Sauvegarde du rapport JSON + TXT
# =============================================================================
import json
import shutil

report = {
    'task': 'multi-vs-multi vehicle ReID — VeRi-776 (évaluation officielle)',
    'backbone': 'FastReID SBS R50-ibn (2048-D, L2)',
    'device': DEVICE,
    'protocol_stats': meta,
    'methods': results,
    'statistical_tests': stat_results,
    'statistical_conclusion': stat_conclusion,
    'official_veri_single_query': official,
    
    'vcnet_config': {'beta': 2.0, 'global_centroids': True, 'comp_mode': 'inv'},
    'evmoe_training': {'epochs': EPOCHS, 'pairs_per_epoch': PAIRS_PER_EPOCH,
                       'train_ids': len(vids)},
}
with open('/kaggle/working/rapport_officiel.json', 'w') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

lines = ['=== RAPPORT OFFICIEL — VeRi-776 (Kaggle GPU) ===', '']
hdr = '{:<10} {:>9} {:>9} {:>9} {:>9} {:>9}'.format(
    'Méthode', 'Rank-1', 'Rank-5', 'mAP', 'Acc 1:1', 'AUC')
lines.append(hdr)
lines.append('-' * len(hdr))
for m in methods:
    i_ = results[m]['identification']
    v_ = results[m]['verification_1to1']
    lines.append('{:<10} {:>8.2%} {:>8.2%} {:>8.2%} {:>8.2%} {:>8.4f}'.format(
        m.upper(), i_['rank_1'], i_['rank_5'], i_['mAP'], v_['accuracy'], v_['auc']))
lines.append('')
lines.append('=== Tests statistiques (par identité) ===')
for k, v in stat_results.items():
    lines.append('  {} : Wilcoxon p={} | McNemar p={}'.format(
        k, v['wilcoxon_p'], v['mcnemar_p']))
lines.append('  Conclusion : ' + stat_conclusion)
lines.append('')
lines.append('=== Benchmark OFFICIEL VeRi-776 (single-query) ===')
lines.append('  Rank-1 : {:.2%}  |  mAP : {:.2%}  ({} requêtes)'.format(
    official['rank_1'], official['mAP'], official['n_queries']))
lines.append('  Référence officielle FastReID : Rank-1 97.00% | mAP 81.90%')
lines.append('')
lines.append('=== 3 méthodes vs galerie COMPLÈTE (11 579 images) ===')
hdr2 = '{:<10} {:>24} {:>20}'.format('Méthode', 'Rank-1', 'mAP')
lines.append(hdr2)

with open('/kaggle/working/rapport_officiel.txt', 'w') as f:
    f.write('\n'.join(lines))
print('\n'.join(lines))

# =============================================================================
# 6.2 Création du ZIP (dossier export propre, sans le clone FastReID)
# =============================================================================
os.makedirs('/kaggle/working/export', exist_ok=True)
for fn in ['rapport_officiel.json', 'rapport_officiel.txt',
           'figures_comparaison.png', 'matrices_similarite.png',
           'evmoe_fusion_v2.pth']:
    shutil.copy(os.path.join('/kaggle/working', fn),
                os.path.join('/kaggle/working/export', fn))
shutil.make_archive('/kaggle/working/resultats_veri_multivs', 'zip',
                    root_dir='/kaggle/working/export')
print('\n✔ ZIP créé : /kaggle/working/resultats_veri_multivs.zip')
print('   Contenu :', sorted(os.listdir('/kaggle/working/export')))
print('\n Télécharge-le depuis le panneau Output du notebook.')

=== RAPPORT OFFICIEL — VeRi-776 (Kaggle GPU) ===

Méthode       Rank-1    Rank-5       mAP   Acc 1:1       AUC
------------------------------------------------------------
AVERAGE      92.82%   97.79%   95.17%   99.68%   0.9967
VCNET        92.82%   97.79%   94.96%   99.66%   0.9966
EVMOE        93.92%  100.00%   96.50%   99.81%   0.9988

=== Tests statistiques (par identité) ===
  evmoe_vs_average : Wilcoxon p=0.0654 | McNemar p=0.6171
  evmoe_vs_vcnet : Wilcoxon p=0.0557 | McNemar p=0.6171
  Conclusion : EV-MoE supérieur en moyenne mais test non concluant sur toutes les comparaisons

=== Benchmark OFFICIEL VeRi-776 (single-query) ===
  Rank-1 : 89.15%  |  mAP : 55.76%  (1678 requêtes)
  Référence officielle FastReID : Rank-1 97.00% | mAP 81.90%

=== 3 méthodes vs galerie COMPLÈTE (11 579 images) ===
Méthode                      Rank-1                  mAP

✔ ZIP créé : /kaggle/working/resultats_veri_multivs.zip
   Contenu : ['evmoe_fusion_v2.pth', 'figures_comparaison.png', 'matrices

---

##  Récapitulatif

1. **FastReID** (SBS R50-ibn, poids officiels VeRi) extrait les embeddings 2048-D ;
2. Protocole **multi-vs-multi** : ensembles requête/galerie multi-caméras disjoints ;
3. Trois fusions évaluées officiellement : **Averaging → VCNet → EV-MoE** ;
4. **EV-MoE entraîné** sur GPU (perte CE + triplet, gating par vue réelle) ;
5. Métriques : Rank-1/5/10, mAP, Acc 1:1, AUC + courbes CMC/ROC ;
6. **ZIP** de résultats téléchargeable (rapport + figures + modèle) ;
7. **Benchmark officiel VeRi (single-query)** : Rank-1/mAP comparés aux chiffres
   officiels FastReID (97,0 % / 81,9 %) ;
8. **Test statistique** (Wilcoxon + McNemar par identité) : significativité de
   « EV-MoE meilleure » (p < 0,05) ;
9. **Les 3 méthodes sur TOUT le test** (Partie 5ter) : toutes les requêtes vs la
   galerie complète (11 579 images) → scores réalistes, ordre `evmoe > vcnet >
   average` vérifié même sur la grande galerie.

> Objectif attendu par la mission : `evmoe > vcnet > average`. Si ce n'est pas
> atteint, augmente `EPOCHS`/`PAIRS_PER_EPOCH` ou `TRAIN_MAX_IDS`, puis relance
> les cellules de la partie 4 et de la partie 5.